# ITCHI con tres radios de atribución: ROCLOUD, W06 (PA21) y Chavas

Casos: Cristóbal 2020 (NA) y Odile 2014 (EP); un caso por corrida (`CASE`).

Todo es igual entre modos salvo el radio exterior de atribución:

- `rocloud`: ROCLOUD por cuadrante.
- `pa21`: W06 con metodología PA21, R(2 m/s), simétrico.
- `chavas`: Chavas desde el R34 observado, R(2 m/s), simétrico (`tcwindprofile` si está instalado; si no, E04 propio).

Radio directo (R34 observado), viento (perfil de `wind.py`), precipitación (MSWEP V2.8 de 3 h en el tiempo válido; sensibilidad con el acumulado de 6 h) y climatología (`p_wet` diario por píxel) son comunes.

In [ ]:
import sys
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from itchi.aggregation import compute_event_accumulated, compute_event_max
from itchi.climatology import extract_standard_percentiles
from itchi.config import get_project_root, load_default_config
from itchi.constants import QUADRANTS
from itchi.mswep import (
    extract_mswep_precipitation,
    infer_mswep_lat_lon_names,
    read_mswep_dataset,
    standardize_mswep_time,
    subset_mswep_domain,
)
from itchi.pipeline import compute_itchi_snapshot_from_grid
from itchi.plotting import plot_itchi_field
from itchi.tracks import DEFAULT_R34_COLUMN_MAP

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "notebooks"))
import itchi.itchi_size_helpers as H  # noqa: E402

from dataclasses import replace

from itchi.plotting import (
    MapStyle,
    configure_map_axis,
    plot_itchi_field,
    plot_itchi_siat_levels,
)


## 0. Parámetros

In [ ]:
CFG = load_default_config()
CASES = {
    "CRISTOBAL2020": dict(name="CRISTOBAL", season=2020, basin="NA", atcf="AL032020",
                          t_map="2020-06-04 12:00"),
    "ODILE2014": dict(name="ODILE", season=2014, basin="EP", atcf="EP152014",
                      t_map="2014-09-15 00:00"),
}
CASE = "CRISTOBAL2020"
C = CASES[CASE]
T_MAP = pd.Timestamp(C["t_map"])            # tiempo para los mapas por snapshot

IBTRACS_FILE = ROOT / CFG["paths"]["tracks"] / f"IBTrACS.{C['basin']}.v04r01.nc"
ROCLOUD_FILE = ROOT / "data" / "external" / "Rocloud" / f"ROCLOUD_{C['basin']}.csv"

MSWEP_3H_DIR = Path("/mnt/d/MSWEP_V280/merged/3hourly")
MSWEP_3H_PATTERN = "{time:%Y%j}.{time:%H}.nc"
MSWEP_3H_STAMP = "start"                    # "start": archivo t cubre [t, t+3h); "end": (t-3h, t]
PRECIP_VAR = CFG["precipitation"]["variable_name"]
WINDOWS_H = (3, 6)                          # 3 h: ITCHI v0.1; 6 h: tiempo válido + 3 h previas
PAD_DEG = 12.0

CLIM_FILE = ROOT / "data" / "processed" / "mswep_percentiles_may_nov.nc"
CLIM_VAR = "p_wet"

MODES = ("rocloud", "rocloud_mean", "pa21", "chavas")
REFERENCE = "rocloud"
COMMON_TIMES_ONLY = True                    # comparar modos sobre los mismos snapshots

FIG_DIR = ROOT / CFG["outputs"]["figures"] / f"itchi_attribution_{CASE.lower()}"
DIAG_DIR = ROOT / CFG["outputs"]["diagnostics"] / "itchi_attribution"
for d in (FIG_DIR, DIAG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Trayectoria y radios (IBTrACS, W06 PA21, Chavas, ROCLOUD)

In [ ]:
track = H.load_case_track(IBTRACS_FILE, ROCLOUD_FILE, C["name"], C["season"], C["atcf"])
track = H.add_chavas_radii(track)            # imprime firma/salida de tcwindprofile si existe
print("fuente Chavas:", track["chavas_source"].value_counts().to_dict())

available = pd.DataFrame({
    mode: [H.attribution_radii_km(row, mode)[0] is not None for _, row in track.iterrows()]
    for mode in MODES
}, index=track["time"])
common = available.all(axis=1)
print("snapshots disponibles por modo:", available.sum().to_dict(), "| comunes:", int(common.sum()))
track_eval = track[common.to_numpy()] if COMMON_TIMES_ONLY else track
track[["time", "vmax_kt", "r34_obs_km", "pa21_r2_km", "chavas_r2_km", "rocloud_mean_km"]].round(1)

## 2. MSWEP de 3 h

In [ ]:
def mswep_3h_path(time: pd.Timestamp) -> Path:
    return MSWEP_3H_DIR / MSWEP_3H_PATTERN.format(time=time)


def window_file_times(valid_time: pd.Timestamp, hours: int) -> list[pd.Timestamp]:
    offset = pd.Timedelta(hours=3) if MSWEP_3H_STAMP == "start" else pd.Timedelta(0)
    ends = pd.date_range(valid_time - pd.Timedelta(hours=hours) + pd.Timedelta(hours=3),
                         valid_time, freq="3h")
    return [end - offset for end in ends]


bbox = dict(
    lon_min=float(track["lon"].min()) - PAD_DEG, lon_max=float(track["lon"].max()) + PAD_DEG,
    lat_min=max(float(track["lat"].min()) - PAD_DEG, -60.0),
    lat_max=min(float(track["lat"].max()) + PAD_DEG, 60.0),
)
FIELD_CACHE: dict = {}


def read_3h_field(file_time: pd.Timestamp) -> xr.DataArray:
    if file_time not in FIELD_CACHE:
        data = read_mswep_dataset(mswep_3h_path(file_time))
        precip = standardize_mswep_time(extract_mswep_precipitation(data, PRECIP_VAR))
        FIELD_CACHE[file_time] = subset_mswep_domain(precip, **bbox).isel(
            valid_time=0, drop=True).load()
    return FIELD_CACHE[file_time]


def window_precipitation(valid_time: pd.Timestamp, hours: int) -> np.ndarray:
    fields = [read_3h_field(t) for t in window_file_times(valid_time, hours)]
    return np.sum([np.asarray(f.values, dtype=float) for f in fields], axis=0)


needed = sorted({t for vt in track["time"] for w in WINDOWS_H for t in window_file_times(vt, w)})
missing = [t for t in needed if not mswep_3h_path(t).exists()]
print(f"{len(needed)} archivos de 3 h; faltan {len(missing)}", missing[:3])

P0 = read_3h_field(needed[0])
LAT_NAME, LON_NAME = infer_mswep_lat_lon_names(P0)
LON2D, LAT2D = np.meshgrid(P0[LON_NAME].values, P0[LAT_NAME].values)
DLAT = abs(float(np.diff(P0[LAT_NAME].values[:2])[0]))
DLON = abs(float(np.diff(P0[LON_NAME].values[:2])[0]))
CELL_AREA_KM2 = (np.deg2rad(DLAT) * H.EARTH_RADIUS_KM) * (
    np.deg2rad(DLON) * H.EARTH_RADIUS_KM * np.cos(np.deg2rad(LAT2D)))
print(P0.shape, "| unidades:", P0.attrs.get("units"))

## 3. Climatología diaria por píxel (`p_wet`)

In [ ]:
clim_ds = xr.open_dataset(CLIM_FILE)
pw = clim_ds[CLIM_VAR]
qdim = next(d for d in pw.dims if d not in infer_mswep_lat_lon_names(pw))
levels = np.asarray(pw[qdim].values, dtype=float)
scale = 100.0 if levels.max() <= 1.0 else 1.0
clim = xr.Dataset({f"Q{int(round(v * scale))}": pw.sel({qdim: v}, drop=True) for v in levels})
clim_lat, clim_lon = infer_mswep_lat_lon_names(clim)
clim = subset_mswep_domain(clim.rename({clim_lat: LAT_NAME, clim_lon: LON_NAME}), **bbox)
clim = clim.reindex_like(P0, method="nearest", tolerance=1e-3)

Q90, Q95, Q99 = (np.asarray(q.values, dtype=float) for q in extract_standard_percentiles(clim))
assert Q90.shape == LON2D.shape
print(f"cobertura {CLIM_VAR}: {np.isfinite(Q95).mean():.1%} | unidades:", pw.attrs.get("units"))

## 4. ITCHI por modo de atribución

In [ ]:
FIELDS = ("ITCHI", "H_dir", "H_ind", "H_W", "H_P", "M_direct", "M_indirect")


def run_snapshot(row: pd.Series, precip: np.ndarray, mode: str):
    radii, n_fixed = H.attribution_radii_km(row, mode)
    if radii is None:
        return None, None, n_fixed
    rmw = float(row["rmw_km"]) if np.isfinite(row["rmw_km"]) else float(row["pa21_rmax_km"])
    fallback = float(row["pa21_r34_km"]) if np.isfinite(row["pa21_r34_km"]) else None
    result = compute_itchi_snapshot_from_grid(
        precipitation=precip, q90=Q90, q95=Q95, q99=Q99,
        lon=LON2D, lat=LAT2D, center_lon=float(row["lon"]), center_lat=float(row["lat"]),
        r34_by_quadrant={q: row[DEFAULT_R34_COLUMN_MAP[q]] for q in QUADRANTS}, r34_unit="nm",
        rocloud_by_quadrant=radii, rocloud_unit="km",
        vmax_kt=float(row["vmax_kt"]), rmw_km=rmw, fallback_direct_radius_km=fallback,
        run_quality_control=True,
    )
    return result, radii, n_fixed


def run_event(mode: str, window_h: int, rows: pd.DataFrame = None):
    rows = track_eval if rows is None else rows
    fields, times, log = [], [], []
    for _, row in rows.iterrows():
        result, radii, n_fixed = run_snapshot(row, window_precipitation(row["time"], window_h), mode)
        log.append(dict(time=row["time"], mode=mode, window_h=window_h, valid=result is not None,
                        quadrants_fixed=n_fixed,
                        attribution_mean_km=np.mean(list(radii.values())) if radii else np.nan))
        if result is not None:
            fields.append({k: np.asarray(result[k]) for k in FIELDS})
            times.append(row["time"])
    dims = ("time", LAT_NAME, LON_NAME)
    coords = {"time": times, LAT_NAME: P0[LAT_NAME].values, LON_NAME: P0[LON_NAME].values}
    ds = xr.Dataset({k: (dims, np.stack([f[k] for f in fields])) for k in FIELDS}, coords=coords)
    ds["ITCHI_max"] = compute_event_max(ds["ITCHI"], dim="time")
    ds["ITCHI_acc"] = compute_event_accumulated(ds["ITCHI"], dim="time")
    ds.attrs.update(case=CASE, atcf=C["atcf"], attribution=mode, window_h=window_h,
                    chavas_source=",".join(sorted(set(track["chavas_source"]))))
    return ds, pd.DataFrame(log)


events, logs = {}, []
for w in WINDOWS_H:
    for mode in MODES:
        events[(mode, w)], log = run_event(mode, w)
        logs.append(log)
logs = pd.concat(logs, ignore_index=True)
logs.groupby(["window_h", "mode"])[["valid", "quadrants_fixed"]].sum()

## 5. Mapas del índice

In [ ]:
W0 = WINDOWS_H[0]


def map_axes(n: int, figsize=(17, 4.8)):
    fig, axes = plt.subplots(1, n, figsize=figsize,
                             subplot_kw={"projection": ccrs.PlateCarree()}, squeeze=False)
    axes = axes[0]
    for ax in axes:
        ax.coastlines(lw=0.6)
        ax.set_extent([bbox["lon_min"], bbox["lon_max"], bbox["lat_min"], bbox["lat_max"]],
                      crs=ccrs.PlateCarree())
        ax.plot(track["lon"], track["lat"], "k.-", lw=0.8, ms=3, transform=ccrs.PlateCarree())
    return fig, axes


# ITCHI_max del evento por modo
fig, axes = map_axes(len(MODES))
for ax, mode in zip(axes, MODES):
    plot_itchi_field(events[(mode, W0)]["ITCHI_max"].values, ax, lon=LON2D, lat=LAT2D,
                     crs=ccrs.PlateCarree(), colorbar_label="ITCHI_max")
    ax.set_title(f"ITCHI_max — {mode} ({W0} h)", fontsize=10)
fig.savefig(FIG_DIR / f"itchi_max_modos_{W0}h.png", dpi=150, bbox_inches="tight")

# Diferencias respecto a ROCLOUD
others = [m for m in MODES if m != REFERENCE]
fig, axes = map_axes(len(others), figsize=(12, 4.8))
for ax, mode in zip(axes, others):
    diff = events[(mode, W0)]["ITCHI_max"] - events[(REFERENCE, W0)]["ITCHI_max"]
    m = ax.pcolormesh(LON2D, LAT2D, diff.values, cmap="RdBu_r", vmin=-0.5, vmax=0.5,
                      transform=ccrs.PlateCarree())
    fig.colorbar(m, ax=ax, shrink=0.8)
    ax.set_title(f"ΔITCHI_max ({mode} − {REFERENCE})", fontsize=10)
fig.savefig(FIG_DIR / f"itchi_max_diferencias_{W0}h.png", dpi=150, bbox_inches="tight")

In [ ]:
# Un snapshot: ITCHI por modo con el alcance geográfico de R34 y del radio de atribución
row = track.loc[track["time"] == T_MAP].squeeze()
fig, axes = map_axes(len(MODES))
for ax, mode in zip(axes, MODES):
    ev = events[(mode, W0)]
    if T_MAP not in ev.indexes["time"]:
        ax.set_title(f"{mode}: sin snapshot en {T_MAP:%Y-%m-%d %H}", fontsize=9)
        continue
    plot_itchi_field(ev["ITCHI"].sel(time=T_MAP).values, ax, lon=LON2D, lat=LAT2D,
                     crs=ccrs.PlateCarree(), colorbar_label="ITCHI")
    radii, _ = H.attribution_radii_km(row, mode)
    lon_o, lat_o = H.quadrant_outline(row["lat"], row["lon"], radii)
    ax.plot(lon_o, lat_o, "m-", lw=1.4, transform=ccrs.PlateCarree(), label=f"atribución {mode}")
    lon_d, lat_d = H.quadrant_outline(row["lat"], row["lon"], H.direct_radii_km(row))
    ax.plot(lon_d, lat_d, "k-", lw=1.0, transform=ccrs.PlateCarree(), label="radio directo")
    ax.plot(row["lon"], row["lat"], "k*", ms=9, transform=ccrs.PlateCarree())
    ax.set_title(f"ITCHI {T_MAP:%Y-%m-%d %H} UTC — {mode}", fontsize=10)
    ax.legend(fontsize=7, loc="lower left")
# fig.savefig(FIG_DIR / f"itchi_snapshot_{T_MAP:%Y%m%d%H}_{W0}h.png", dpi=150, bbox_inches="tight")

In [ ]:
SNAP_TIME = pd.Timestamp(C["t_map"])        # posición a mapear (cámbiala libremente)
SNAP_WINDOW_H = WINDOWS_H[1]                # 3 h
SNAP_MODES = MODES                          # ("rocloud", "pa21", "chavas")
SNAP_FIELDS = ( "H_dir", "H_ind", "ITCHI", "SIAT")
SNAP_HALF_WIDTH_DEG = 14.0
SNAP_STYLE = MapStyle(
    lon_tick_spacing=8.0, lat_tick_spacing=8.0, grid_spacing=4.0,
    grid_color="w", grid_alpha=0.6, label_size=10.0,
)
PC = ccrs.PlateCarree()


In [ ]:



def snapshot_style(row: pd.Series, base: MapStyle = SNAP_STYLE,
                   half_width: float = SNAP_HALF_WIDTH_DEG) -> MapStyle:
    # Extensión centrada en el ciclón, alineada a múltiplos del espaciado de ticks
    step = base.lon_tick_spacing
    lon0 = np.floor((row["lon"] - half_width) / step) * step
    lon1 = np.ceil((row["lon"] + half_width) / step) * step
    lat0 = np.floor((row["lat"] - 0.75 * half_width) / step) * step
    lat1 = np.ceil((row["lat"] + 0.75 * half_width) / step) * step
    return replace(base, lon_extent=(lon0, lon1), lat_extent=(lat0, lat1))


def overlay_radii(ax, row: pd.Series, attribution: dict | None) -> None:
    ax.plot(track["lon"], track["lat"], "k-", lw=0.8, transform=PC, zorder=5)
    x, y = H.quadrant_outline(row["lat"], row["lon"], H.direct_radii_km(row))
    ax.plot(x, y, "k-", lw=1.4, transform=ccrs.Geodetic(), zorder=6, label="radio directo")
    if attribution is not None:
        x, y = H.quadrant_outline(row["lat"], row["lon"], attribution)
        ax.plot(x, y, "m-", lw=1.6, transform=ccrs.Geodetic(), zorder=6, label="atribución")
    ax.plot(row["lon"], row["lat"], "k*", ms=10, transform=PC, zorder=7)


def plot_position_snapshot(time: pd.Timestamp, window_h: int = SNAP_WINDOW_H,
                           modes=SNAP_MODES, fields=SNAP_FIELDS):
    row = track.loc[track["time"] == time]
    assert len(row) == 1, f"sin posición en {time}"
    row = row.iloc[0]
    precip = window_precipitation(time, window_h)
    style = snapshot_style(row)

    fig, axes = plt.subplots(len(modes), len(fields),
                             figsize=(4.8 * len(fields), 4.2 * len(modes)),
                             subplot_kw={"projection": PC}, squeeze=False)
    for i, mode in enumerate(modes):
        result, radii, _ = run_snapshot(row, precip, mode)
        for j, field in enumerate(fields):
            ax = configure_map_axis(axes[i, j], crs=PC, style=style)
            if result is None:
                ax.set_title(f"{mode}: sin radio de atribución válido", fontsize=9)
                continue
            if field == "P":
                wet = np.where(precip >= 0.1, precip, np.nan)
                mesh = ax.pcolormesh(LON2D, LAT2D, wet, cmap="YlGnBu", shading="auto",
                                     transform=PC, zorder=1)
                plt.colorbar(mesh, ax=ax, shrink=0.8, label=f"P ({window_h} h, mm)")
                ax.contour(LON2D, LAT2D, precip - Q95, levels=[0.0], colors="r",
                           linewidths=0.7, transform=PC, zorder=5)   # P = Q95 diario
            elif field == "SIAT":
                plot_itchi_siat_levels(result["ITCHI"], ax, lon=LON2D, lat=LAT2D, crs=PC,
                                       colorbar_label="nivel ITCHI", zorder=1)
            else:
                values = np.asarray(result[field], dtype=float)
                plot_itchi_field(np.where(values > 0, values, np.nan), ax, lon=LON2D,
                                 lat=LAT2D, crs=PC, colorbar_label=field, zorder=1)
            overlay_radii(ax, row, radii)
            ax.set_title(f"{field} — {mode}", fontsize=11)
    axes[0, 0].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{CASE} {time:%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt | "
                 f"MSWEP {window_h} h", fontsize=13)
    fig.tight_layout()
    return fig


# fig = plot_position_snapshot(SNAP_TIME)
# # fig.savefig(FIG_DIR / f"snapshot_{SNAP_TIME:%Y%m%d%H}_{SNAP_WINDOW_H}h.png",
# #             dpi=150, bbox_inches="tight")

In [ ]:
from itchi.plotting import SIAT_DANGER_LABELS
import warnings 

CBAR = dict(height=0.70, width=0.035, pad=0.03, labelsize=9)   # fracciones del eje del mapa


def add_small_colorbar(ax, mappable, label: str, ticks=None, ticklabels=None, **opts):
    # Colorbar en un eje insertado: alto = fracción del alto del mapa, centrado
    o = {**CBAR, **opts}
    cax = ax.inset_axes([1.0 + o["pad"], (1.0 - o["height"]) / 2, o["width"], o["height"]])
    cbar = ax.figure.colorbar(mappable, cax=cax, ticks=ticks)
    cbar.set_label(label, fontsize=o["labelsize"])
    cbar.ax.tick_params(labelsize=o["labelsize"] - 1)
    if ticklabels is not None:
        cbar.set_ticklabels(ticklabels)
    return cbar


def draw_field(ax, field: str, result: dict, precip: np.ndarray, row: pd.Series,
               radii: dict | None, window_h: int, cbar_opts: dict | None = None):
    cbar_opts = cbar_opts or {}
    if field == "P":
        wet = np.where(precip >= 0.1, precip, np.nan)
        mesh = ax.pcolormesh(LON2D, LAT2D, wet, cmap="YlGnBu", shading="auto",
                             transform=PC, zorder=1)
        add_small_colorbar(ax, mesh, f"P ({window_h} h, mm)", **cbar_opts)
        ax.contour(LON2D, LAT2D, precip - Q95, levels=[0.0], colors="r",
                   linewidths=0.7, transform=PC, zorder=5)            # P = Q95 diario
    elif field == "SIAT":
        mesh = plot_itchi_siat_levels(result["ITCHI"], ax, lon=LON2D, lat=LAT2D, crs=PC,
                                      add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, "nivel ITCHI", ticks=np.arange(0.5, 6.5, 1.0),
                           ticklabels=[lab.split(" ", 1)[0] for lab in SIAT_DANGER_LABELS],
                           **cbar_opts)
    else:
        values = np.asarray(result[field], dtype=float)
        mesh = plot_itchi_field(np.where(values > 0, values, np.nan), ax, lon=LON2D,
                                lat=LAT2D, crs=PC, add_colorbar=False, zorder=1)
        add_small_colorbar(ax, mesh, field, **cbar_opts)
    overlay_radii(ax, row, radii)


def plot_position_snapshot(time: pd.Timestamp, window_h: int = SNAP_WINDOW_H,
                           modes=SNAP_MODES, fields=SNAP_FIELDS):
    # Versión en rejilla (modos x campos) con colorbars pequeñas
    row = track.loc[track["time"] == time].iloc[0]
    precip = window_precipitation(time, window_h)
    style = snapshot_style(row)
    fig, axes = plt.subplots(len(modes), len(fields),
                             figsize=(5.0 * len(fields), 4.2 * len(modes)),
                             subplot_kw={"projection": PC}, squeeze=False)
    fig.subplots_adjust(wspace=0.45, hspace=0.25)
    for i, mode in enumerate(modes):
        
        result, radii, _ = run_snapshot(row, precip, mode)
        for j, field in enumerate(fields):
            ax = configure_map_axis(axes[i, j], crs=PC, style=style)
            if result is None:
                ax.set_title(f"{mode}: sin radio de atribución válido", fontsize=9)
                continue
            draw_field(ax, field, result, precip, row, radii, window_h)
            ax.set_title(f"{field} — {mode}", fontsize=11)
    axes[0, 0].legend(fontsize=8, loc="lower left")
    fig.suptitle(f"{CASE} {time:%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt | "
                 f"MSWEP {window_h} h", fontsize=13)
    return fig
            

def plot_position_field(time: pd.Timestamp, mode: str, field: str,
                        window_h: int = SNAP_WINDOW_H, half_width: float = SNAP_HALF_WIDTH_DEG,
                        figsize=(9, 7), style: MapStyle | None = None, cbar_opts=None):
    # Un solo mapa a detalle: una posición, un modo de atribución, un campo
    row = track.loc[track["time"] == time].iloc[0]
    precip = window_precipitation(time, window_h)
    result, radii, _ = run_snapshot(row, precip, mode)
    if result is None:
        warnings.warn(f"{mode}: sin radio de atribución válido en {time}; se omite el mapa")
        return None
    base = style or replace(SNAP_STYLE, label_size=12.0, tick_size=7.0)
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": PC})
    configure_map_axis(ax, crs=PC, style=snapshot_style(row, base, half_width))
    draw_field(ax, field, result, precip, row, radii, window_h,
               cbar_opts={"labelsize": 11, **(cbar_opts or {})})
    ax.legend(fontsize=9, loc="lower left")
    ax.set_title(f"{field} — atribución {mode}\n{CASE} {time:%Y-%m-%d %H} UTC | "
                 f"Vmax {row['vmax_kt']:.0f} kt | MSWEP {window_h} h", fontsize=12)
    return fig

In [ ]:
# Rejilla completa con barras pequeñas
fig = plot_position_snapshot(SNAP_TIME)
# fig.savefig(FIG_DIR / f"snapshot_{SNAP_TIME:%Y%m%d%H}.png", dpi=150, bbox_inches="tight")



In [ ]:
# Un mapa a detalle por campo y modo
# fig = plot_position_field(SNAP_TIME, mode="chavas", field="ITCHI")
fig = plot_position_field(SNAP_TIME, mode="rocloud", field="H_ind", half_width=10)
fig = plot_position_field(SNAP_TIME, mode="pa21", field="P", window_h=6,
                          cbar_opts={"height": 0.5})

# Todos los campos y modos de una posición, cada uno en su propia figura
for mode in MODES:
    for field in SNAP_FIELDS:
        fig = plot_position_field(SNAP_TIME, mode, field)
        if fig is None:
            continue
        fig.savefig(FIG_DIR / f"{SNAP_TIME:%Y%m%d%H}_{mode}_{field}.png",
                    dpi=150, bbox_inches="tight")
        plt.close(fig)

## 6. Métricas del índice por modo

Áreas en km² (celdas ponderadas por cos φ). Las métricas de diferencia se calculan sobre la unión de las huellas (ITCHI_max > 0 en el modo o en la referencia).

In [ ]:
def event_metrics(ev: xr.Dataset, ref: xr.Dataset) -> dict:
    a = CELL_AREA_KM2
    x = np.nan_to_num(ev["ITCHI_max"].values)
    y = np.nan_to_num(ref["ITCHI_max"].values)
    direct = ev["M_direct"].any("time").values
    indirect = ev["M_indirect"].any("time").values & ~direct
    h_ind = np.nan_to_num(ev["H_ind"].max("time").values)
    union = (x > 0) | (y > 0)
    d, w = (x - y)[union], a[union]
    both, either = (x >= 0.5) & (y >= 0.5), (x >= 0.5) | (y >= 0.5)
    footprint = x > 0
    return {
        "snapshots": int(ev.sizes["time"]),
        "area_directa_km2": a[direct].sum(),
        "area_indirecta_km2": a[indirect].sum(),
        "area_ITCHI>=0.25_km2": a[x >= 0.25].sum(),
        "area_ITCHI>=0.50_km2": a[x >= 0.50].sum(),
        "area_ITCHI>=0.75_km2": a[x >= 0.75].sum(),
        "ITCHI_integrado_km2": (x * a).sum(),
        "ITCHI_max_medio_en_huella": (x * a)[footprint].sum() / a[footprint].sum(),
        "H_ind_medio_en_indirecta": (h_ind * a)[indirect].sum() / max(a[indirect].sum(), 1e-9),
        "dITCHI_medio_vs_ref": np.average(d, weights=w) if w.sum() else np.nan,
        "RMSD_vs_ref": np.sqrt(np.average(d**2, weights=w)) if w.sum() else np.nan,
        "frac_area_|d|>0.1": w[np.abs(d) > 0.1].sum() / w.sum() if w.sum() else np.nan,
        "Jaccard_ITCHI>=0.5": a[both].sum() / a[either].sum() if a[either].sum() else np.nan,
    }


rows = []
for (mode, w), ev in events.items():
    rows.append({"case": CASE, "window_h": w, "mode": mode,
                 **event_metrics(ev, events[(REFERENCE, w)])})
metrics = pd.DataFrame(rows).set_index(["case", "window_h", "mode"])
metrics.to_csv(DIAG_DIR / f"metricas_evento_{CASE.lower()}.csv")
metrics.round(3)

In [ ]:
# Métricas por snapshot: área indirecta e ITCHI integrado
per_time = []
for (mode, w), ev in events.items():
    a = CELL_AREA_KM2
    for t in ev["time"].values:
        snap = ev.sel(time=t)
        indirect = snap["M_indirect"].values & ~snap["M_direct"].values
        per_time.append(dict(case=CASE, window_h=w, mode=mode, time=pd.Timestamp(t),
                             area_indirecta_km2=a[indirect].sum(),
                             ITCHI_integrado_km2=(np.nan_to_num(snap["ITCHI"].values) * a).sum(),
                             H_ind_max=float(np.nanmax(snap["H_ind"].values))))
per_time = pd.DataFrame(per_time).merge(
    logs[["time", "mode", "window_h", "attribution_mean_km"]], on=["time", "mode", "window_h"])
per_time.to_csv(DIAG_DIR / f"metricas_snapshot_{CASE.lower()}.csv", index=False)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for mode in MODES:
    d = per_time[(per_time["mode"] == mode) & (per_time["window_h"] == W0)]
    axes[0].plot(d["time"], d["attribution_mean_km"], label=mode)
    axes[1].plot(d["time"], d["area_indirecta_km2"] / 1e3, label=mode)
    axes[2].plot(d["time"], d["ITCHI_integrado_km2"] / 1e3, label=mode)
axes[0].set_ylabel("radio de atribución\nmedio (km)")
axes[1].set_ylabel("área indirecta\n(10³ km²)")
axes[2].set_ylabel("ITCHI integrado\n(10³ km²)")
axes[0].legend()
fig.autofmt_xdate()
fig.savefig(FIG_DIR / f"metricas_snapshot_{W0}h.png", dpi=150, bbox_inches="tight")

## 7. Tabla conjunta (tras correr ambos casos)

In [ ]:
paths = sorted(DIAG_DIR.glob("metricas_evento_*.csv"))
pd.concat([pd.read_csv(p) for p in paths]).round(3) if paths else None